# 02 — Feature Engineering

Transform raw insurance-quote data into agent-ready features: date-based urgency,
actuarial risk tiers, ordinal/label encodings, and per-agent feature sets.
Outputs are saved as Parquet splits and reusable encoder artefacts.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pathlib import Path

DATA_PATH = Path("../datasets/insurance_quotes.csv")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")

## 1. Date Features

In [ ]:
for col in ["Q_Creation_DT", "Q_Valid_DT", "Policy_Bind_DT"]:
    df[col] = pd.to_datetime(df[col], format="%Y/%m/%d", errors="coerce")

df["urgency_days"] = (df["Q_Valid_DT"] - df["Q_Creation_DT"]).dt.days
# Guard against dirty date rows: negative or null urgency gets median fallback.
df.loc[df["urgency_days"] < 0, "urgency_days"] = np.nan
urgency_median = int(df["urgency_days"].median())
df["urgency_days"] = df["urgency_days"].fillna(urgency_median).astype(int)

print("urgency_days stats:")
print(df["urgency_days"].describe())
print(f"\nMedian fallback used: {urgency_median}")
print(f"\nNull counts:")
print(df[["Q_Creation_DT", "Q_Valid_DT", "Policy_Bind_DT", "urgency_days"]].isnull().sum())

## 2. Risk Tier Engineering (Agent 1 Target)

No ground-truth risk label exists. We engineer `risk_tier` from actuarial signals.

In [ ]:
MILES_RISK = {
    "<= 7.5 K": 0,
    "> 7.5 K & <= 15 K": 1,
    "> 15 K & <= 25 K": 2,
    "> 25 K & <= 35 K": 3,
    "> 35 K & <= 45 K": 4,
    "> 45 K & <= 55 K": 5,
    "> 55 K": 6,
}

USAGE_RISK = {"Pleasure": 0, "Commute": 5, "Business": 10}


def compute_risk_score(row):
    score = 0.0
    score += row["Prev_Accidents"] * 30
    score += row["Prev_Citations"] * 20

    age = row["Driver_Age"]
    if age < 25:
        score += 25
    elif age < 30:
        score += 10
    elif age > 70:
        score += 15
    elif age > 60:
        score += 5

    exp = row["Driving_Exp"]
    if exp < 3:
        score += 20
    elif exp < 5:
        score += 12
    elif exp < 10:
        score += 5

    score += MILES_RISK.get(row["Annual_Miles_Range"], 2)
    score += USAGE_RISK.get(row["Veh_Usage"], 3)
    return score


df["risk_score"] = df.apply(compute_risk_score, axis=1)

p50 = df["risk_score"].quantile(0.50)
p85 = df["risk_score"].quantile(0.85)
print(f"Percentile thresholds — p50: {p50}, p85: {p85}")

df["risk_tier"] = pd.cut(
    df["risk_score"],
    bins=[-1, p50, p85, df["risk_score"].max() + 1],
    labels=["LOW", "MEDIUM", "HIGH"],
)

print("\nRisk tier distribution:")
print(df["risk_tier"].value_counts())
print("\nRisk tier proportions:")
print(df["risk_tier"].value_counts(normalize=True).round(4))
print("\nCross-tab risk_tier vs Policy_Bind:")
print(pd.crosstab(df["risk_tier"], df["Policy_Bind"], margins=True))

## 3. Categorical Encoding (for Agent 2 — LightGBM)

In [ ]:
COVERAGE_MAP = {"Basic": 0, "Balanced": 1, "Enhanced": 2}
AGENT_TYPE_MAP = {"EA": 0, "IA": 1}

SAL_ORDER = [
    "<= $ 25 K",
    "> $ 25 K <= $ 40 K",
    "> $ 40 K <= $ 60 K",
    "> $ 60 K <= $ 90 K",
    "> $ 90 K ",
]
SAL_MAP = {v: i for i, v in enumerate(SAL_ORDER)}

VEHICLE_COST_ORDER = [
    "<= $ 10 K",
    "> $ 10 K <= $ 20 K",
    "> $ 20 K <= $ 30 K",
    "> $ 30 K <= $ 40 K",
    "> $ 40 K ",
]
VEHICLE_COST_MAP = {v: i for i, v in enumerate(VEHICLE_COST_ORDER)}

RISK_TIER_MAP = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}

df["Coverage_enc"] = df["Coverage"].map(COVERAGE_MAP).fillna(0).astype(int)
df["Agent_Type_enc"] = df["Agent_Type"].map(AGENT_TYPE_MAP).fillna(0).astype(int)
df["Sal_Range_enc"] = df["Sal_Range"].map(SAL_MAP).fillna(0).astype(int)
df["Vehicl_Cost_enc"] = df["Vehicl_Cost_Range"].map(VEHICLE_COST_MAP).fillna(0).astype(int)
df["Annual_Miles_enc"] = df["Annual_Miles_Range"].map(MILES_RISK).fillna(2).astype(int)
df["Veh_Usage_enc"] = df["Veh_Usage"].map(USAGE_RISK).fillna(3).astype(int)
df["Re_Quote_enc"] = (df["Re_Quote"] == "Yes").astype(int)
df["Policy_Bind_enc"] = (df["Policy_Bind"] == "Yes").astype(int)

label_encoders = {}
for col in ["Region", "Gender", "Marital_Status", "Education", "Policy_Type"]:
    le = LabelEncoder()
    df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

df["risk_tier_enc"] = df["risk_tier"].astype(str).map(RISK_TIER_MAP).fillna(1).astype(int)

ordinal_maps = {
    "COVERAGE_MAP": COVERAGE_MAP,
    "AGENT_TYPE_MAP": AGENT_TYPE_MAP,
    "SAL_MAP": SAL_MAP,
    "VEHICLE_COST_MAP": VEHICLE_COST_MAP,
    "MILES_RISK": MILES_RISK,
    "USAGE_RISK": USAGE_RISK,
    "RISK_TIER_MAP": RISK_TIER_MAP,
}

joblib.dump(label_encoders, MODELS_DIR / "label_encoders.joblib")
joblib.dump(ordinal_maps, MODELS_DIR / "ordinal_maps.joblib")

enc_cols = [c for c in df.columns if c.endswith("_enc")]
print(f"Encoded columns ({len(enc_cols)}): {enc_cols}")
print(f"\nNull check on encoded columns:")
print(df[enc_cols].isnull().sum())

## 4. Feature Sets Per Agent

In [ ]:
AGENT1_FEATURES = [
    "Prev_Accidents",
    "Prev_Citations",
    "Driver_Age",
    "Driving_Exp",
    "HH_Vehicles",
    "HH_Drivers",
    "Annual_Miles_Range",
    "Veh_Usage",
]

AGENT1_CAT_FEATURES = ["Annual_Miles_Range", "Veh_Usage"]

AGENT2_FEATURES = [
    "Re_Quote_enc",
    "urgency_days",
    "Coverage_enc",
    "Agent_Type_enc",
    "Region_enc",
    "Sal_Range_enc",
    "HH_Drivers",
    "HH_Vehicles",
    "Quoted_Premium",
    "Vehicl_Cost_enc",
    "Driver_Age",
    "Driving_Exp",
    "Prev_Accidents",
    "Prev_Citations",
    "Gender_enc",
    "Marital_Status_enc",
    "Education_enc",
    "risk_tier_enc",
]

print(f"Agent 1 features ({len(AGENT1_FEATURES)}): {AGENT1_FEATURES}")
print(f"Agent 1 cat features ({len(AGENT1_CAT_FEATURES)}): {AGENT1_CAT_FEATURES}")
print(f"Agent 2 features ({len(AGENT2_FEATURES)}): {AGENT2_FEATURES}")

## 5. Train/Test Split

In [ ]:
df["Annual_Miles_Range"] = df["Annual_Miles_Range"].fillna("<= 7.5 K").astype(str)
df["Veh_Usage"] = df["Veh_Usage"].fillna("Commute").astype(str)

all_needed = list(set(AGENT1_FEATURES + AGENT2_FEATURES + ["Policy_Bind_enc", "risk_tier", "risk_tier_enc", "risk_score"]))
df_clean = df.dropna(subset=all_needed)
print(f"Rows after dropping nulls in feature columns: {len(df_clean):,} (dropped {len(df) - len(df_clean):,})")

train_df, test_df = train_test_split(
    df_clean,
    test_size=0.20,
    random_state=42,
    stratify=df_clean["Policy_Bind_enc"],
)

print(f"\nTrain: {len(train_df):,} rows  |  Test: {len(test_df):,} rows")
print(f"Train bind rate: {train_df['Policy_Bind_enc'].mean():.4f}")
print(f"Test  bind rate: {test_df['Policy_Bind_enc'].mean():.4f}")

train_df.to_parquet(MODELS_DIR / "train.parquet", index=False)
test_df.to_parquet(MODELS_DIR / "test.parquet", index=False)

feature_config = {
    "AGENT1_FEATURES": AGENT1_FEATURES,
    "AGENT1_CAT_FEATURES": AGENT1_CAT_FEATURES,
    "AGENT2_FEATURES": AGENT2_FEATURES,
    "RISK_TIER_MAP": RISK_TIER_MAP,
}
joblib.dump(feature_config, MODELS_DIR / "feature_config.joblib")

print(f"\nSaved to {MODELS_DIR.resolve()}:")
for f in sorted(MODELS_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")